In [1]:
import duckdb
from run_query import connect_to_repo, run_sql
 
con = connect_to_repo()
print("DuckDB", duckdb.__version__)

DuckDB 2.0.0


In [2]:
run_sql("""
CREATE OR REPLACE TABLE students (roll_no INTEGER, name VARCHAR,
                                  block VARCHAR, cgpa DOUBLE);
INSERT INTO students VALUES
    (101, 'Asha', 'A', 8.4), (102, 'Ravi', 'B', 6.9),
    (103, 'Meera', 'A', 9.1), (104, 'Kiran', NULL, 7.5);
SELECT * FROM students;
""", con)

┌─────────┬─────────┬─────────┬────────┐
│ roll_no │  name   │  block  │  cgpa  │
│  int32  │ varchar │ varchar │ double │
├─────────┼─────────┼─────────┼────────┤
│     101 │ Asha    │ A       │    8.4 │
│     102 │ Ravi    │ B       │    6.9 │
│     103 │ Meera   │ A       │    9.1 │
│     104 │ Kiran   │ NULL    │    7.5 │
└─────────┴─────────┴─────────┴────────┘



In [3]:
run_sql("""
SELECT name, cgpa
FROM students
WHERE cgpa > 7
ORDER BY cgpa DESC
LIMIT 2;
""", con)

┌─────────┬────────┐
│  name   │  cgpa  │
│ varchar │ double │
├─────────┼────────┤
│ Meera   │    9.1 │
│ Asha    │    8.4 │
└─────────┴────────┘



In [4]:
run_sql("""
DESCRIBE SELECT * FROM read_csv('data/raw/adverse_events_10k.csv');
""", con)

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ safety_report_id │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ drug_name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ reaction         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ serious          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ patient_age      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ patient_age_unit │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ country          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ event_date       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [5]:
run_sql("""
CREATE OR REPLACE VIEW events AS
SELECT *
FROM read_csv('data/raw/adverse_events_10k.csv',
              types = {'safety_report_id': 'VARCHAR',
                       'serious': 'VARCHAR',
                       'patient_age_unit': 'VARCHAR',
                       'event_date': 'VARCHAR'});
SELECT COUNT(*) AS reports FROM events;
""", con)

┌─────────┐
│ reports │
│  int64  │
├─────────┤
│    6218 │
└─────────┘



In [6]:
run_sql("""
SELECT drug_name, reaction, serious, country, event_date
FROM events
LIMIT 3;
""", con)

┌───────────┬───────────────────────┬─────────┬─────────┬────────────┐
│ drug_name │       reaction        │ serious │ country │ event_date │
│  varchar  │        varchar        │ varchar │ varchar │  varchar   │
├───────────┼───────────────────────┼─────────┼─────────┼────────────┤
│ ARANESP   │ Product storage error │ 2       │ US      │ 20240701   │
│ ENDARI    │ Cholecystectomy       │ 1       │ NULL    │ 20240701   │
│ ENDARI    │ Osteonecrosis         │ 1       │ NULL    │ 20240701   │
└───────────┴───────────────────────┴─────────┴─────────┴────────────┘



In [7]:
run_sql("""
CREATE OR REPLACE VIEW drug_ref AS
SELECT * FROM read_csv('data/raw/drug_reference.csv');
CREATE OR REPLACE VIEW countries AS
SELECT * FROM read_csv('data/raw/country_codes.csv');
SELECT (SELECT COUNT(*) FROM drug_ref)  AS drug_ref_rows,
       (SELECT COUNT(*) FROM countries) AS country_rows;
""", con)

┌───────────────┬──────────────┐
│ drug_ref_rows │ country_rows │
│     int64     │    int64     │
├───────────────┼──────────────┤
│           521 │           54 │
└───────────────┴──────────────┘



In [8]:
run_sql("""
SELECT COUNT(*) AS serious_japan
FROM events
WHERE country = 'JP'
  AND serious = '1';
""", con)

┌───────────────┐
│ serious_japan │
│     int64     │
├───────────────┤
│           161 │
└───────────────┘



In [9]:
run_sql("""
SELECT COUNT(*) AS lower_case_match
FROM events
WHERE drug_name = 'dupixent';
""", con)

┌──────────────────┐
│ lower_case_match │
│      int64       │
├──────────────────┤
│                0 │
└──────────────────┘



In [10]:
run_sql("""
SELECT COUNT(*) AS no_brackets
FROM events
WHERE country = 'JP' OR country = 'CA'
  AND serious = '1';
""", con)

┌─────────────┐
│ no_brackets │
│    int64    │
├─────────────┤
│         346 │
└─────────────┘



In [11]:
run_sql("""
SELECT COUNT(*) AS with_brackets
FROM events
WHERE (country = 'JP' OR country = 'CA')
  AND serious = '1';
""", con)

┌───────────────┐
│ with_brackets │
│     int64     │
├───────────────┤
│           321 │
└───────────────┘



In [12]:
run_sql("""
SELECT COUNT(*) AS with_in
FROM events
WHERE country IN ('JP', 'CA')
  AND serious = '1';
""", con)

┌─────────┐
│ with_in │
│  int64  │
├─────────┤
│     321 │
└─────────┘



In [13]:
run_sql("""
SELECT COUNT(*)                  AS reports,
       COUNT(DISTINCT drug_name) AS different_drugs,
       COUNT(DISTINCT country)   AS different_countries
FROM events;
""", con)

┌─────────┬─────────────────┬─────────────────────┐
│ reports │ different_drugs │ different_countries │
│  int64  │      int64      │        int64        │
├─────────┼─────────────────┼─────────────────────┤
│    6218 │            1125 │                  82 │
└─────────┴─────────────────┴─────────────────────┘



In [14]:
run_sql("""
SELECT name FROM students WHERE block = NULL;
SELECT name FROM students WHERE block IS NULL;
""", con)

-- Result 1
┌─────────┐
│  name   │
│ varchar │
└─────────┘
  0 rows 

-- Result 2
┌─────────┐
│  name   │
│ varchar │
├─────────┤
│ Kiran   │
└─────────┘



In [15]:
run_sql("""
SELECT COUNT(*)           AS reports,
       COUNT(country)     AS with_country,
       COUNT(patient_age) AS with_age,
       COUNT(serious)     AS with_seriousness
FROM events;
""", con)

┌─────────┬──────────────┬──────────┬──────────────────┐
│ reports │ with_country │ with_age │ with_seriousness │
│  int64  │    int64     │  int64   │      int64       │
├─────────┼──────────────┼──────────┼──────────────────┤
│    6218 │         5815 │     3202 │             6205 │
└─────────┴──────────────┴──────────┴──────────────────┘



In [16]:
run_sql("""
SELECT COALESCE(country, 'Not reported') AS country, COUNT(*) AS reports
FROM events
WHERE country IS NULL OR country = 'US'
GROUP BY COALESCE(country, 'Not reported')
ORDER BY reports DESC;
""", con)

┌──────────────┬─────────┐
│   country    │ reports │
│   varchar    │  int64  │
├──────────────┼─────────┤
│ US           │    4234 │
│ Not reported │     403 │
└──────────────┴─────────┘



In [17]:
run_sql("""
SELECT 7 / 2           AS seven_halves,
       5 / 0           AS five_by_zero,
       0 / 0           AS zero_by_zero,
       5 / NULLIF(0, 0) AS safe_division;
""", con)

┌──────────────┬──────────────┬──────────────┬───────────────┐
│ seven_halves │ five_by_zero │ zero_by_zero │ safe_division │
│    double    │    double    │    double    │    double     │
├──────────────┼──────────────┼──────────────┼───────────────┤
│          3.5 │          inf │         -nan │          NULL │
└──────────────┴──────────────┴──────────────┴───────────────┘



In [18]:
run_sql("""
SELECT country,
       COUNT(*)                  AS reports,
       COUNT(DISTINCT drug_name) AS drugs,
       ROUND(AVG(patient_age), 1) AS avg_age_as_recorded
FROM events
WHERE country IS NOT NULL
GROUP BY country
ORDER BY reports DESC
LIMIT 5;
""", con)

┌─────────┬─────────┬───────┬─────────────────────┐
│ country │ reports │ drugs │ avg_age_as_recorded │
│ varchar │  int64  │ int64 │       double        │
├─────────┼─────────┼───────┼─────────────────────┤
│ US      │    4234 │   694 │                54.6 │
│ GB      │     253 │   119 │                56.7 │
│ EU      │     229 │   148 │                57.0 │
│ JP      │     186 │   117 │                45.0 │
│ CA      │     170 │    79 │                56.3 │
└─────────┴─────────┴───────┴─────────────────────┘



In [19]:
try:
    run_sql("""
    SELECT country, drug_name, COUNT(*) AS reports
    FROM events
    GROUP BY country;
    """, con)
except duckdb.Error as error:
    print(type(error).__name__, "-", str(error).splitlines()[0])

BinderException - Binder Error: column "drug_name" must appear in the GROUP BY clause or must be part of an aggregate function.


In [20]:
try:
    run_sql("""
    SELECT country, COUNT(*) AS reports
    FROM events
    WHERE COUNT(*) >= 100
    GROUP BY country;
    """, con)
except duckdb.Error as error:
    print(type(error).__name__, "-", str(error).splitlines()[0])

BinderException - Binder Error: WHERE clause cannot contain aggregates!


In [21]:
run_sql("""
SELECT COALESCE(country, 'Not reported') AS country, COUNT(*) AS reports
FROM events
GROUP BY COALESCE(country, 'Not reported')
HAVING COUNT(*) >= 150
ORDER BY reports DESC;
""", con)

┌──────────────┬─────────┐
│   country    │ reports │
│   varchar    │  int64  │
├──────────────┼─────────┤
│ US           │    4234 │
│ Not reported │     403 │
│ GB           │     253 │
│ EU           │     229 │
│ JP           │     186 │
│ CA           │     170 │
└──────────────┴─────────┘



In [22]:
run_sql("""
SELECT country,
       COUNT(*) AS reports,
       SUM(CASE WHEN serious = '1' THEN 1 ELSE 0 END) AS serious,
       ROUND(100.0 * SUM(CASE WHEN serious = '1' THEN 1 ELSE 0 END)
             / COUNT(serious), 1) AS pct_serious
FROM events
WHERE country IN ('US', 'GB', 'JP')
GROUP BY country
ORDER BY reports DESC;
""", con)

┌─────────┬─────────┬─────────┬─────────────┐
│ country │ reports │ serious │ pct_serious │
│ varchar │  int64  │ int128  │   double    │
├─────────┼─────────┼─────────┼─────────────┤
│ US      │    4234 │    1238 │        29.3 │
│ GB      │     253 │     247 │        97.6 │
│ JP      │     186 │     161 │        86.6 │
└─────────┴─────────┴─────────┴─────────────┘



In [23]:
run_sql("""
SELECT event_date,
       strptime(event_date, '%Y%m%d')          AS as_date,
       year(strptime(event_date, '%Y%m%d'))    AS yr,
       quarter(strptime(event_date, '%Y%m%d')) AS qtr
FROM events
LIMIT 2;
""", con)

┌────────────┬─────────────────────┬───────┬───────┐
│ event_date │       as_date       │  yr   │  qtr  │
│  varchar   │      timestamp      │ int64 │ int64 │
├────────────┼─────────────────────┼───────┼───────┤
│ 20240701   │ 2024-07-01 00:00:00 │  2024 │     3 │
│ 20240701   │ 2024-07-01 00:00:00 │  2024 │     3 │
└────────────┴─────────────────────┴───────┴───────┘



In [24]:
run_sql("""
SELECT year(strptime(event_date, '%Y%m%d')) AS yr,
       COUNT(*) AS reports
FROM events
GROUP BY yr
ORDER BY yr;
""", con)

┌───────┬─────────┐
│  yr   │ reports │
│ int64 │  int64  │
├───────┼─────────┤
│  2024 │    1493 │
│  2025 │    3070 │
│  2026 │    1655 │
└───────┴─────────┘



In [25]:
run_sql("""
CREATE OR REPLACE TABLE marks (roll_no INTEGER, subject VARCHAR,
                               mark INTEGER);
INSERT INTO marks VALUES
    (101, 'SQL', 78), (101, 'Python', 85), (102, 'SQL', 64),
    (103, 'SQL', 91), (105, 'SQL', 70);
SELECT * FROM marks;
""", con)

┌─────────┬─────────┬───────┐
│ roll_no │ subject │ mark  │
│  int32  │ varchar │ int32 │
├─────────┼─────────┼───────┤
│     101 │ SQL     │    78 │
│     101 │ Python  │    85 │
│     102 │ SQL     │    64 │
│     103 │ SQL     │    91 │
│     105 │ SQL     │    70 │
└─────────┴─────────┴───────┘



In [26]:
run_sql("""
SELECT s.roll_no, s.name, m.subject, m.mark
FROM students AS s
INNER JOIN marks AS m ON s.roll_no = m.roll_no
ORDER BY s.roll_no, m.subject;
""", con)

┌─────────┬─────────┬─────────┬───────┐
│ roll_no │  name   │ subject │ mark  │
│  int32  │ varchar │ varchar │ int32 │
├─────────┼─────────┼─────────┼───────┤
│     101 │ Asha    │ Python  │    85 │
│     101 │ Asha    │ SQL     │    78 │
│     102 │ Ravi    │ SQL     │    64 │
│     103 │ Meera   │ SQL     │    91 │
└─────────┴─────────┴─────────┴───────┘



In [27]:
run_sql("""
SELECT 'LEFT' AS join_type, s.name, m.roll_no AS marks_roll, m.mark
FROM students AS s LEFT JOIN marks AS m ON s.roll_no = m.roll_no
WHERE m.roll_no IS NULL
UNION ALL
SELECT 'RIGHT', s.name, m.roll_no, m.mark
FROM students AS s RIGHT JOIN marks AS m ON s.roll_no = m.roll_no
WHERE s.roll_no IS NULL;
""", con)

┌───────────┬─────────┬────────────┬───────┐
│ join_type │  name   │ marks_roll │ mark  │
│  varchar  │ varchar │   int32    │ int32 │
├───────────┼─────────┼────────────┼───────┤
│ LEFT      │ Kiran   │       NULL │  NULL │
│ RIGHT     │ NULL    │        105 │    70 │
└───────────┴─────────┴────────────┴───────┘



In [28]:
run_sql("""
SELECT
 (SELECT COUNT(*) FROM students s INNER JOIN marks m
     ON s.roll_no = m.roll_no)                        AS inner_rows,
 (SELECT COUNT(*) FROM students s LEFT JOIN marks m
     ON s.roll_no = m.roll_no)                        AS left_rows,
 (SELECT COUNT(*) FROM students s RIGHT JOIN marks m
     ON s.roll_no = m.roll_no)                        AS right_rows,
 (SELECT COUNT(*) FROM students s FULL OUTER JOIN marks m
     ON s.roll_no = m.roll_no)                        AS full_rows;
""", con)

┌────────────┬───────────┬────────────┬───────────┐
│ inner_rows │ left_rows │ right_rows │ full_rows │
│   int64    │   int64   │   int64    │   int64   │
├────────────┼───────────┼────────────┼───────────┤
│          4 │         5 │          5 │         6 │
└────────────┴───────────┴────────────┴───────────┘



In [29]:
run_sql("""
SELECT COUNT(*)            AS reports,
       COUNT(r.drug_class) AS with_class,
       COUNT(*) - COUNT(r.drug_class) AS without_class
FROM events AS e
LEFT JOIN drug_ref AS r ON e.drug_name = r.drug_name;
""", con)

┌─────────┬────────────┬───────────────┐
│ reports │ with_class │ without_class │
│  int64  │   int64    │     int64     │
├─────────┼────────────┼───────────────┤
│    6218 │       3987 │          2231 │
└─────────┴────────────┴───────────────┘



In [30]:
run_sql("""
CREATE OR REPLACE TABLE drug_ref_dup AS
SELECT * FROM drug_ref
UNION ALL
SELECT * FROM drug_ref WHERE drug_name = 'DUPIXENT';
 
SELECT COUNT(*) AS rows_after_join
FROM events AS e
LEFT JOIN drug_ref_dup AS r ON e.drug_name = r.drug_name;
""", con)

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│            6839 │
└─────────────────┘



In [31]:
run_sql("""
SELECT drug_name, COUNT(*) AS times_listed
FROM drug_ref_dup
GROUP BY drug_name
HAVING COUNT(*) > 1;
""", con)

┌───────────┬──────────────┐
│ drug_name │ times_listed │
│  varchar  │    int64     │
├───────────┼──────────────┤
│ DUPIXENT  │            2 │
└───────────┴──────────────┘



In [32]:
run_sql("""
SELECT COUNT(*)            AS countries,
       ROUND(AVG(reports), 1) AS avg_reports_per_country,
       MAX(reports)        AS biggest
FROM (SELECT country, COUNT(*) AS reports
      FROM events
      WHERE country IS NOT NULL
      GROUP BY country) AS per_country;
""", con)

┌───────────┬─────────────────────────┬─────────┐
│ countries │ avg_reports_per_country │ biggest │
│   int64   │         double          │  int64  │
├───────────┼─────────────────────────┼─────────┤
│        82 │                    70.9 │    4234 │
└───────────┴─────────────────────────┴─────────┘



In [33]:
run_sql("""
SELECT country, COUNT(*) AS reports
FROM events
WHERE country IS NOT NULL
GROUP BY country
HAVING COUNT(*) > (SELECT AVG(reports)
                   FROM (SELECT COUNT(*) AS reports
                         FROM events
                         WHERE country IS NOT NULL
                         GROUP BY country) AS per_country)
ORDER BY reports DESC;
""", con)

┌─────────┬─────────┐
│ country │ reports │
│ varchar │  int64  │
├─────────┼─────────┤
│ US      │    4234 │
│ GB      │     253 │
│ EU      │     229 │
│ JP      │     186 │
│ CA      │     170 │
│ CN      │     129 │
└─────────┴─────────┘



In [34]:
run_sql("""
SELECT m.roll_no, m.subject, m.mark
FROM marks AS m
WHERE m.mark = (SELECT MAX(m2.mark)
                FROM marks AS m2
                WHERE m2.roll_no = m.roll_no)
ORDER BY m.roll_no;
""", con)

┌─────────┬─────────┬───────┐
│ roll_no │ subject │ mark  │
│  int32  │ varchar │ int32 │
├─────────┼─────────┼───────┤
│     101 │ Python  │    85 │
│     102 │ SQL     │    64 │
│     103 │ SQL     │    91 │
│     105 │ SQL     │    70 │
└─────────┴─────────┴───────┘



In [35]:
run_sql("""
SELECT name FROM students
WHERE roll_no NOT IN (SELECT roll_no FROM marks);
""", con)

┌─────────┐
│  name   │
│ varchar │
├─────────┤
│ Kiran   │
└─────────┘

